# What the score is measuring

Pairwise precision and recall use proper-name mentions that match exactly one gold alias. Two mentions are a gold pair when they share that alias's entity. They are a predicted pair when the resolver gave them the same `entity_id`.

Chapters 1–13 are the tune split. Chapters 14–27 are held out. `must_links.yml` is knowledge, so the file reports the held-out score twice: with those links, and with the weights unchanged and the links removed. The bar for v1 is held-out F1 at least 0.90 with the knowledge file.

Read the numbers from `data/kg/eval.json`. Do not copy them into the notebook. A later rebuild should still run this cell.

Read precision and recall from the printout. Precision at 1.0 means every predicted pair agrees with the gold aliases. Recall below 1.0 is a true alias that stayed in a second cluster, such as a bare `John` that did not take the Seward id. The gap between the with-knowledge score and the without-knowledge score is `must_links.yml`. The review queue (`data/kg/review_queue.csv`, gitignored) is the 0.60–0.85 band. Lowering 0.85 would mix people who share a surname.

`n_relations` counts pattern edges. Zero is a successful refusal. Widening the pattern window to a whole sentence attached "married" to the wrong neighbour, including Jonathan to Peter Hawkins.

`n_stated_relations` counts the relationships in `data/gold/relations.yml`. Each row carries a quote copied from the novel ("I want to see your patient, Mr. Renfield."), and the build fails if the quote is not in one document. An earlier version looked for one sentence with both names and a cue word close to each. It confirmed only 4 of 19 rows, because the novel spreads a relationship over a paragraph: Lucy's letter names "Dr. John Seward" several sentences before he asks if she "could love him in time". Rows without a quote still go through that check. The curated Carfax `part_of` London edge is neither kind; it comes from `configs/kg.yml`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import json

eval_path = ROOT / "data" / "kg" / "eval.json"
report = json.loads(eval_path.read_text(encoding="utf-8"))
for key in (
    "tune_with_knowledge",
    "heldout_with_knowledge",
    "heldout_without_knowledge",
    "nominals",
):
    print(key)
    print(report[key])
    print()
print("n_relations", report["n_relations"])
print("relation rows", report.get("relations", []))


## Nominals

Title nominals such as `the Count` attach only when exactly one cluster in that chapter already carries the honorific. An unresolved rate near one half means the rule declined the ambiguous chapters. That is the same preference as the kinship patterns: skip the guess.
